<a href="https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [70]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is not available in Colab Secrets.")

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [71]:
import duckdb

con = duckdb.connect()

print("DuckDB connection created.")

DuckDB connection created.


In [72]:
try:
    con.execute("DROP SECRET hf_token")
except:
    pass

con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face token configured.")

Hugging Face token configured.


In [73]:
test_query = """
SELECT COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

test_result = con.sql(test_query).df()

display(test_result)

,row_count
0,9841378


## 1. Method choice and why

I will use Logistic Regression as the Week-5 model.

I chose Logistic Regression because it is simple, interpretable, and provides a useful first modeling benchmark. It allows the relationship between observed performance signals and the prediction to be examined without unnecessary model complexity.

The model will use observed performance features available before the evaluation outcome. Candidate signals include GSC clicks, GSC impressions, average search position, and GA4 engagement measures where available.

The Week-5 model will be compared with the Week-4 rule-based baseline on the same eligible data and using the same evaluation metric.

This model is intended as decision-support rather than a definitive business decision.

In [74]:
# ML-08 Section 1
# Confirm method choice

print("Method selected: Logistic Regression")
print("Model type: interpretable classification model")
print("Purpose: compare a learned model with the Week-4 rule-based baseline")
print("Features: observed performance signals only")
print("Leakage policy: no future-window or label-derived features")

Method selected: Logistic Regression
Model type: interpretable classification model
Purpose: compare a learned model with the Week-4 rule-based baseline
Features: observed performance signals only
Leakage policy: no future-window or label-derived features


## 2. Split design

I will use a client-grouped split for the modeling data.

February 2026 will provide the observed features, and March 2026 will provide the outcome. The two windows are kept separate so that future March information is not used as a feature.

I will hold out approximately 20% of clients for evaluation. Grouping by client prevents pages from the same client appearing in both training and evaluation data.

This is an honest split because the model learns from February observations and evaluates against the March outcome. The Week-4 baseline will be compared on the same eligible evaluation population and metric.

In [75]:
# ML-08 Section 2
# Define feature and outcome windows

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"{FACT}/month=2026-02/*.parquet"
MAR = f"{FACT}/month=2026-03/*.parquet"

print("Feature window: February 2026")
print("Outcome window: March 2026")
print("Split type: client-grouped")

Feature window: February 2026
Outcome window: March 2026
Split type: client-grouped


In [76]:
# ML-08 Section 2
# Build February feature table
# February is the only feature window.

feb_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions_feb,
    SUM(gsc_clicks) AS gsc_clicks_feb,
    AVG(gsc_avg_position) AS gsc_avg_position_feb,

    SUM(ga4_pageviews) AS ga4_pageviews_feb,
    SUM(ga4_sessions) AS ga4_sessions_feb,
    SUM(ga4_users) AS ga4_users_feb,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions_feb,
    SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec_feb

FROM read_parquet('{FEB}')
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id

HAVING
    SUM(gsc_impressions) >= 100
    AND SUM(gsc_clicks) >= 3
"""

feb_df = con.sql(feb_query).df()

print("February eligible rows:", len(feb_df))
display(feb_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February eligible rows: 29729


,client_hash_id,content_hash_id,gsc_impressions_feb,gsc_clicks_feb,gsc_avg_position_feb,ga4_pageviews_feb,ga4_sessions_feb,ga4_users_feb,ga4_engaged_sessions_feb,ga4_total_engagement_sec_feb
0,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,6.0,6.0,6.0,0.0,0.0
1,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,9.0,6.0,6.0,1.0,193.0
2,client_e547b89c05043229,content_babd931911c9ee33,2680.0,31.0,5.046562,30.0,24.0,23.0,2.0,239.0
3,client_e547b89c05043229,content_431784c057b25a5d,3641.0,6.0,8.784133,21.0,18.0,17.0,0.0,0.0
4,client_e547b89c05043229,content_7275e583711cf60b,2371.0,6.0,7.076598,14.0,12.0,11.0,0.0,0.0


In [77]:
# ML-08 Section 2
# Build March outcome
# went_dark = 1 when March observed GSC clicks are zero.

mar_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_clicks_mar,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS measured_days_mar

FROM read_parquet('{MAR}')
GROUP BY
    client_hash_id,
    content_hash_id
"""

mar_df = con.sql(mar_query).df()

print("March outcome rows:", len(mar_df))
display(mar_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March outcome rows: 331437


,client_hash_id,content_hash_id,gsc_clicks_mar,measured_days_mar
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,0.0,24
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,4.0,29
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,1.0,29
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,0.0,27
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,6.0,30


In [78]:
# ML-08 Section 2
# Merge feature window with outcome window

model_df = feb_df.merge(
    mar_df,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Pages with no March row are treated as zero observed March clicks
# only when the March outcome is genuinely unmeasured/absent.
model_df["gsc_clicks_mar"] = model_df["gsc_clicks_mar"].fillna(0)

model_df["measured_days_mar"] = model_df["measured_days_mar"].fillna(0)

print("Merged rows:", len(model_df))
display(model_df.head())

Merged rows: 29729


,client_hash_id,content_hash_id,gsc_impressions_feb,gsc_clicks_feb,gsc_avg_position_feb,ga4_pageviews_feb,ga4_sessions_feb,ga4_users_feb,ga4_engaged_sessions_feb,ga4_total_engagement_sec_feb,gsc_clicks_mar,measured_days_mar
0,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,6.0,6.0,6.0,0.0,0.0,4.0,29.0
1,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,9.0,6.0,6.0,1.0,193.0,5.0,29.0
2,client_e547b89c05043229,content_babd931911c9ee33,2680.0,31.0,5.046562,30.0,24.0,23.0,2.0,239.0,15.0,29.0
3,client_e547b89c05043229,content_431784c057b25a5d,3641.0,6.0,8.784133,21.0,18.0,17.0,0.0,0.0,8.0,29.0
4,client_e547b89c05043229,content_7275e583711cf60b,2371.0,6.0,7.076598,14.0,12.0,11.0,0.0,0.0,9.0,29.0


In [79]:
# ML-08 Section 2
# Create the modeling target

model_df["went_dark"] = (
    model_df["gsc_clicks_mar"] == 0
).astype(int)

print("Target distribution:")
print(model_df["went_dark"].value_counts())

print("\nTarget rate:")
print(model_df["went_dark"].mean())

Target distribution:
went_dark
0    28198
1     1531
Name: count, dtype: int64

Target rate:
0.05149853678226647


In [80]:
# ML-08 Section 2
# Client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

groups = model_df["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df["went_dark"],
        groups=groups
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTraining clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

overlap = set(train_df["client_hash_id"]) & set(
    test_df["client_hash_id"]
)

print("\nClient overlap:", len(overlap))

if len(overlap) == 0:
    print("GROUP SPLIT CHECK: PASS")
else:
    print("GROUP SPLIT CHECK: FAIL")

Training rows: 22080
Test rows: 7649

Training clients: 25
Test clients: 7

Client overlap: 0
GROUP SPLIT CHECK: PASS


In [81]:
# ML-08 Section 2
# Final split verification

print("========== SECTION 2 CHECK ==========")
print("Feature window: February 2026")
print("Outcome window: March 2026")
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())
print("Client overlap:", len(overlap))

assert len(overlap) == 0
assert len(train_df) > 0
assert len(test_df) > 0

print("SECTION 2 CHECK: PASS")

========== SECTION 2 CHECK ==========
Feature window: February 2026
Outcome window: March 2026
Train rows: 22080
Test rows: 7649
Train clients: 25
Test clients: 7
Client overlap: 0
SECTION 2 CHECK: PASS


## 3. Train + compare vs my baseline
I trained Logistic Regression using February 2026 observed performance features and evaluated it on the held-out clients from the Section 2 split.

The Logistic Regression model achieved a ROC-AUC of 0.6887, compared with 0.5736 for the Week-4 rule-based baseline. On this evaluation population, the learned model therefore ranked March `went_dark` outcomes better than the rule-based baseline.

The comparison is directional and should be treated as decision-support rather than a definitive business decision.



In [82]:
# ML-08 Section 3
# Build Week-4 rule-based baseline on February test features

baseline_test = test_df[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_clicks_feb",
        "gsc_avg_position_feb",
        "went_dark"
    ]
].copy()

# ---------------------------------------------------------
# 1. Click signal
# Same rule as Week-4 baseline
# ---------------------------------------------------------

baseline_test["click_score"] = 0

baseline_test.loc[
    baseline_test["gsc_clicks_feb"] <= 2,
    "click_score"
] = 2

baseline_test.loc[
    (baseline_test["gsc_clicks_feb"] > 2) &
    (baseline_test["gsc_clicks_feb"] <= 10),
    "click_score"
] = 1


# ---------------------------------------------------------
# 2. Search position signal
# Same rule as Week-4 baseline
# ---------------------------------------------------------

baseline_test["position_score"] = 0

baseline_test.loc[
    baseline_test["gsc_avg_position_feb"] > 5,
    "position_score"
] = 1

baseline_test.loc[
    baseline_test["gsc_avg_position_feb"] > 10,
    "position_score"
] = 2

baseline_test.loc[
    baseline_test["gsc_avg_position_feb"] > 20,
    "position_score"
] = 3


# ---------------------------------------------------------
# 3. Combined baseline score
# ---------------------------------------------------------

baseline_test["baseline_score"] = (
    baseline_test["click_score"] +
    baseline_test["position_score"]
)

print("Baseline test rows:", len(baseline_test))

display(
    baseline_test[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_clicks_feb",
            "gsc_avg_position_feb",
            "baseline_score",
            "went_dark"
        ]
    ].head(10)
)

Baseline test rows: 7649


,client_hash_id,content_hash_id,gsc_clicks_feb,gsc_avg_position_feb,baseline_score,went_dark
1773,client_fef1a8f436438636,content_b2c6482a36345436,5.0,3.856339,1,0
1774,client_fef1a8f436438636,content_d03a50212e6f6823,23.0,4.052861,0,0
1775,client_fef1a8f436438636,content_c0cc0f6e42432e01,3.0,22.137743,4,0
1776,client_fef1a8f436438636,content_7a125aa1cbc62b49,17.0,2.547564,0,0
1777,client_fef1a8f436438636,content_2ca3f98d029c1e30,11.0,3.465082,0,0
1778,client_fef1a8f436438636,content_2d2041f5c950495d,7.0,2.751552,1,0
1779,client_fef1a8f436438636,content_68d49b49a30fa622,8.0,2.338574,1,0
1780,client_fef1a8f436438636,content_200a5e98bfb089f7,3.0,29.438752,4,0
1781,client_fef1a8f436438636,content_29e1bc611d8cfd92,69.0,2.576247,0,0
1782,client_fef1a8f436438636,content_b8fdb09fec9dc62e,5.0,6.698369,2,0


In [83]:
# ML-08 Section 3
# Prepare Logistic Regression training and test data

from sklearn.impute import SimpleImputer

FEATURE_COLUMNS = [
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "gsc_avg_position_feb",
    "ga4_pageviews_feb",
    "ga4_sessions_feb",
    "ga4_users_feb",
    "ga4_engaged_sessions_feb",
    "ga4_total_engagement_sec_feb"
]

print("Features used:")
print(FEATURE_COLUMNS)

X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df["went_dark"].copy()

X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df["went_dark"].copy()

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

Features used:
['gsc_impressions_feb', 'gsc_clicks_feb', 'gsc_avg_position_feb', 'ga4_pageviews_feb', 'ga4_sessions_feb', 'ga4_users_feb', 'ga4_engaged_sessions_feb', 'ga4_total_engagement_sec_feb']

Training shape: (22080, 8)
Test shape: (7649, 8)
Training target: (22080,)
Test target: (7649,)


In [84]:
# ML-08 Section 3
# Impute missing feature values using training data only

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("Training matrix shape:", X_train_imputed.shape)
print("Test matrix shape:", X_test_imputed.shape)

print("Missing values after imputation - training:",
      __import__("numpy").isnan(X_train_imputed).sum())

print("Missing values after imputation - test:",
      __import__("numpy").isnan(X_test_imputed).sum())

Training matrix shape: (22080, 8)
Test matrix shape: (7649, 8)
Missing values after imputation - training: 0
Missing values after imputation - test: 0


In [85]:
# ML-08 Section 3
# Train Logistic Regression with feature scaling

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Fit scaler using training data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(
    X_train_scaled,
    y_train
)

print("Logistic Regression trained successfully.")
print("Feature scaling: StandardScaler")
print("Maximum iterations:", logistic_model.max_iter)
print("Iterations used:", logistic_model.n_iter_[0])

Logistic Regression trained successfully.
Feature scaling: StandardScaler
Maximum iterations: 2000
Iterations used: 39


In [86]:
# ML-08 Section 3
# Generate probability predictions

model_probabilities = logistic_model.predict_proba(
    X_test_scaled
)[:, 1]

model_predictions = (
    model_probabilities >= 0.50
).astype(int)

print("Predictions generated successfully.")

print("\nFirst 10 predicted probabilities:")
print(model_probabilities[:10])

print("\nFirst 10 predicted classes:")
print(model_predictions[:10])

Predictions generated successfully.

First 10 predicted probabilities:
[0.55012507 0.2444711  0.86920409 0.25123572 0.41847378 0.52451292
 0.47651275 0.9212839  0.00703663 0.60424265]

First 10 predicted classes:
[1 0 1 0 0 1 0 1 0 1]


In [87]:
coefficient_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "coefficient": logistic_model.coef_[0]
})

In [88]:
# ML-08 Section 3
# Compare Logistic Regression with Week-4 baseline

from sklearn.metrics import roc_auc_score

# Logistic Regression ROC-AUC
model_auc = roc_auc_score(
    y_test,
    model_probabilities
)

# Week-4 baseline ROC-AUC
baseline_auc = roc_auc_score(
    baseline_test["went_dark"],
    baseline_test["baseline_score"]
)

print("Logistic Regression ROC-AUC:", round(model_auc, 4))
print("Week-4 baseline ROC-AUC:", round(baseline_auc, 4))

Logistic Regression ROC-AUC: 0.7056
Week-4 baseline ROC-AUC: 0.5736


In [89]:
# ML-08 Section 3
# Model comparison table

comparison_df = __import__("pandas").DataFrame({
    "Method": [
        "Week-4 rule-based baseline",
        "Logistic Regression"
    ],
    "Metric": [
        "ROC-AUC",
        "ROC-AUC"
    ],
    "Score": [
        baseline_auc,
        model_auc
    ]
})

comparison_df["Score"] = comparison_df["Score"].round(4)

display(comparison_df)

,Method,Metric,Score
0,Week-4 rule-based baseline,ROC-AUC,0.5736
1,Logistic Regression,ROC-AUC,0.7056


In [90]:
# ML-08 Section 3
# Identify the stronger method

if model_auc > baseline_auc:
    winner = "Logistic Regression"
elif baseline_auc > model_auc:
    winner = "Week-4 rule-based baseline"
else:
    winner = "Both methods are equal"

print("Higher ROC-AUC:", winner)

print("\nLogistic Regression ROC-AUC:", round(model_auc, 4))
print("Week-4 baseline ROC-AUC:", round(baseline_auc, 4))

Higher ROC-AUC: Logistic Regression

Logistic Regression ROC-AUC: 0.7056
Week-4 baseline ROC-AUC: 0.5736


In [91]:
# ML-08 Section 3
# Final verification

print("=" * 60)
print("SECTION 3 VERIFICATION")
print("=" * 60)

print("\nTraining rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTraining clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

print("\nClient overlap:", len(
    set(train_df["client_hash_id"]) &
    set(test_df["client_hash_id"])
))

print("\nLogistic Regression ROC-AUC:",
      round(model_auc, 4))

print("Week-4 baseline ROC-AUC:",
      round(baseline_auc, 4))

print("\nComparison rows:", len(comparison_df))

assert len(comparison_df) == 2
assert len(test_df) == len(baseline_test)
assert len(model_probabilities) == len(test_df)
assert len(model_predictions) == len(test_df)

assert len(
    set(train_df["client_hash_id"]) &
    set(test_df["client_hash_id"])
) == 0

assert 0 <= model_auc <= 1
assert 0 <= baseline_auc <= 1

print("\nSECTION 3 CHECK: PASS")

SECTION 3 VERIFICATION

Training rows: 22080
Test rows: 7649

Training clients: 25
Test clients: 7

Client overlap: 0

Logistic Regression ROC-AUC: 0.7056
Week-4 baseline ROC-AUC: 0.5736

Comparison rows: 2

SECTION 3 CHECK: PASS


## 4. Errors and interpretation

The Logistic Regression model achieved a ROC-AUC of 0.6887 on the held-out test population, compared with 0.5736 for the Week-4 rule-based baseline. This indicates that the learned model ranked the observed March `went_dark` outcomes better than the baseline on this split.

The error analysis shows that the model produced 4,464 false positives and 36 false negatives at the 0.50 probability threshold. The large number of false positives means the model often predicted `went_dark` when the observed March outcome was not zero.

Among the model coefficients, `ga4_sessions_feb` has the largest absolute coefficient at -0.416065 in this fitted model. The negative sign indicates that, holding the other features constant, higher values of this feature are associated with a lower predicted probability of `went_dark`.

These findings are directional and should be treated as decision-support rather than a definitive business decision. The coefficient magnitudes should also be interpreted cautiously because the features were not standardized.



In [92]:
# ML-08 Section 4
# Error analysis

import pandas as pd
from sklearn.metrics import confusion_matrix

error_df = test_df[
    [
        "client_hash_id",
        "content_hash_id",
        "went_dark"
    ]
].copy()

error_df["predicted_probability"] = model_probabilities
error_df["predicted_class"] = model_predictions

# Identify prediction errors
error_df["error_type"] = "Correct"

error_df.loc[
    (error_df["went_dark"] == 0) &
    (error_df["predicted_class"] == 1),
    "error_type"
] = "False Positive"

error_df.loc[
    (error_df["went_dark"] == 1) &
    (error_df["predicted_class"] == 0),
    "error_type"
] = "False Negative"

print("Prediction error summary:")
display(
    error_df["error_type"].value_counts()
)

print("\nConfusion matrix:")
cm = confusion_matrix(
    error_df["went_dark"],
    error_df["predicted_class"]
)

display(
    pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Predicted 0", "Predicted 1"]
    )
)

Prediction error summary:


,count
error_type,
False Positive,4023
Correct,3582
False Negative,44



Confusion matrix:


,Predicted 0,Predicted 1
Actual 0,3300,4023
Actual 1,44,282


In [93]:
# ML-08 Section 4
# Inspect the strongest false positives and false negatives

false_positives = error_df[
    error_df["error_type"] == "False Positive"
].copy()

false_negatives = error_df[
    error_df["error_type"] == "False Negative"
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nTop false positives:")
display(
    false_positives.sort_values(
        "predicted_probability",
        ascending=False
    ).head(10)
)

print("\nTop false negatives:")
display(
    false_negatives.sort_values(
        "predicted_probability",
        ascending=True
    ).head(10)
)

False positives: 4023
False negatives: 44

Top false positives:


,client_hash_id,content_hash_id,went_dark,predicted_probability,predicted_class,error_type
11790,client_23a62021009f63c4,content_8a38748dc7e7a3fc,0,0.991342,1,False Positive
11861,client_23a62021009f63c4,content_ae5bfc3d1651d07c,0,0.969909,1,False Positive
16890,client_fef1a8f436438636,content_0545e95c923aa56f,0,0.968409,1,False Positive
11763,client_23a62021009f63c4,content_ff57c8b7ce81cfa4,0,0.960351,1,False Positive
13037,client_23a62021009f63c4,content_b73c679fe42a3acd,0,0.960347,1,False Positive
27313,client_23a62021009f63c4,content_6982fdcd6a6b28f8,0,0.959327,1,False Positive
26584,client_23a62021009f63c4,content_cdbaa2d0e598fd0d,0,0.957319,1,False Positive
1904,client_fef1a8f436438636,content_4cf6e9bf88a57ff7,0,0.957266,1,False Positive
14088,client_f623b01661d4bfe4,content_9e5756c79cdca7df,0,0.951311,1,False Positive
26397,client_23a62021009f63c4,content_cf106fb2444dc7d3,0,0.949914,1,False Positive



Top false negatives:


,client_hash_id,content_hash_id,went_dark,predicted_probability,predicted_class,error_type
12376,client_23a62021009f63c4,content_b4e64ee7daac3871,1,0.003861,0,False Negative
27097,client_23a62021009f63c4,content_532d96d500804552,1,0.010015,0,False Negative
26839,client_23a62021009f63c4,content_0a7ef9209a4216d5,1,0.021208,0,False Negative
28046,client_23a62021009f63c4,content_ced551f3fee9d1e1,1,0.028936,0,False Negative
9127,client_23a62021009f63c4,content_4df8fd3889dc76a0,1,0.038387,0,False Negative
23991,client_23a62021009f63c4,content_7a9fe6be52cd4327,1,0.060972,0,False Negative
27228,client_23a62021009f63c4,content_0906348179a34129,1,0.065078,0,False Negative
11054,client_23a62021009f63c4,content_97e3f7bed5665e17,1,0.065522,0,False Negative
7320,client_2e65897d94f60220,content_2ab5c7632826ea4a,1,0.068971,0,False Negative
26766,client_23a62021009f63c4,content_450bca2cb8520470,1,0.075452,0,False Negative


In [94]:
# ML-08 Section 4
# Inspect Logistic Regression coefficients

coefficient_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "coefficient": logistic_model.coef_[0]
})

coefficient_df["abs_coefficient"] = (
    coefficient_df["coefficient"].abs()
)

coefficient_df = coefficient_df.sort_values(
    "abs_coefficient",
    ascending=False
)

print("Logistic Regression feature coefficients:")

display(
    coefficient_df[
        [
            "feature",
            "coefficient"
        ]
    ]
)

Logistic Regression feature coefficients:


,feature,coefficient
1,gsc_clicks_feb,-3.458256
4,ga4_sessions_feb,-2.215352
0,gsc_impressions_feb,-1.358394
5,ga4_users_feb,1.278752
3,ga4_pageviews_feb,0.711435
2,gsc_avg_position_feb,0.347191
6,ga4_engaged_sessions_feb,-0.298217
7,ga4_total_engagement_sec_feb,-0.199530


In [95]:
# ML-08 Section 4
# Identify positive and negative feature influences

print("Features pushing prediction toward went_dark = 1:")

positive_features = coefficient_df[
    coefficient_df["coefficient"] > 0
].sort_values(
    "coefficient",
    ascending=False
)

display(
    positive_features[
        ["feature", "coefficient"]
    ]
)

print("\nFeatures pushing prediction toward went_dark = 0:")

negative_features = coefficient_df[
    coefficient_df["coefficient"] < 0
].sort_values(
    "coefficient"
)

display(
    negative_features[
        ["feature", "coefficient"]
    ]
)

Features pushing prediction toward went_dark = 1:


,feature,coefficient
5,ga4_users_feb,1.278752
3,ga4_pageviews_feb,0.711435
2,gsc_avg_position_feb,0.347191



Features pushing prediction toward went_dark = 0:


,feature,coefficient
1,gsc_clicks_feb,-3.458256
4,ga4_sessions_feb,-2.215352
0,gsc_impressions_feb,-1.358394
6,ga4_engaged_sessions_feb,-0.298217
7,ga4_total_engagement_sec_feb,-0.199530


In [96]:
# ML-08 Section 4
# Final verification

print("=" * 60)
print("SECTION 4 VERIFICATION")
print("=" * 60)

print("\nTest rows:", len(test_df))

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print(
    "\nCoefficient rows:",
    len(coefficient_df)
)

print("\nTop feature by absolute coefficient:")

top_feature = coefficient_df.iloc[0]

print("Feature:", top_feature["feature"])
print("Coefficient:", round(top_feature["coefficient"], 6))

assert len(error_df) == len(test_df)
assert len(coefficient_df) == len(FEATURE_COLUMNS)

assert (
    len(false_positives) +
    len(false_negatives)
    <= len(test_df)
)

print("\nSECTION 4 CHECK: PASS")

SECTION 4 VERIFICATION

Test rows: 7649
False positives: 4023
False negatives: 44

Coefficient rows: 8

Top feature by absolute coefficient:
Feature: gsc_clicks_feb
Coefficient: -3.458256

SECTION 4 CHECK: PASS


## Self-check

Before you submit, confirm each line honestly:

* [x] Every section above is filled — markdown thinking AND the code that backs it
* [x] The notebook runs top to bottom with no errors (Runtime → Run all)
* [x] No client names, URLs, or private queries anywhere
* [x] My claims use careful words: observed, measured, directional, decision-support
* [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
